In [2]:
from __future__ import annotations
import os
import json
import importlib
import argparse
from functools import partial

import random
import numpy as np
import pickle

import torch

from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

from core import test_mini_mcts_search_v55
# from core import mcts_search_mini_v12
from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k

# from core.llm_engine import rm_engine
# from core.llms import rm_generate
# import logging
# logging.basicConfig(format='%(message)s', level=logging.ERROR)

In [3]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [4]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [5]:
# llm_total_gpu = 0.4
# llm_gpu_memory_utilization = 0.2
# # llm_vllm = LLM(
# #     model = llm_dir,
# #     tensor_parallel_size=1,
# #     gpu_memory_utilization = 0.7,  # Utilize 50% of GPU memory
# #     # enable_prefix_caching=True,  # V100 doesn't support enable_prefix_caching 
# #     # enable_chunked_prefill=False, # and enable_chunked_prefill
# #     max_model_len = 5000,
# #     dtype = "float16",
# #     seed = config.seed)

# llm_vllm = LLM(
#     model=llm_dir, 
#     tensor_parallel_size=1, 
#     # trust_remote_code=True,
#     swap_space=16,
#     max_model_len=5000,
#     gpu_memory_utilization=llm_gpu_memory_utilization,
#     enforce_eager=True,
#     distributed_executor_backend=None,
#     dtype="float16",
#     seed=0,
# )

# print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))




KeyboardInterrupt



In [ ]:
# llm_vllm_embeds = LLM(
#     model=llm_dir, 
#     tensor_parallel_size=1, 
#     # trust_remote_code=True,
#     task="embed",
#     swap_space=16,
#     max_model_len=5000,
#     gpu_memory_utilization=llm_total_gpu-llm_gpu_memory_utilization,
#     enforce_eager=True,
#     distributed_executor_backend=None,
#     dtype="float16",
#     seed=0,
# )
# print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

In [ ]:
# prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
# print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

In [22]:
def level_order_rec(cur_node, level, level_thres, res):
    if cur_node is None:
        return 

    if level > level_thres:
        return
    
    if len(res) <= level:
        res.append([])

    res[level].append(cur_node.visit_count())
    for child_node in cur_node.children:
        level_order_rec(child_node, level+1, level_thres, res)

def level_order(root, level_thres):
    res = []
    level_order_rec(root, 0, level_thres, res)
    return res

In [ ]:
stop

In [6]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)

1: 43
2: 90
3: 105
4: 128
5: 134


In [7]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 3            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0

# mcts parameter
config.num_batches = 1
config.batch_budget = config.num_batches*config.max_depths 
config.num_phases = config.batch_budget

config.lam = 10 
config.normalize_embeds = True

config.cpuct_root = 0
config.cpuct_leaf = 0
config.cpuct = 10000
config.ds_beta = 1.0
config.ds_alpha = 10.0
config.use_ppl = True

config.version = "v51"



In [9]:
level = 4                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
num_questions = 1
num_trials = 1
print(f"num_questions = {num_questions}")
print(f"num_trials = {num_trials}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
q_idx = 9
batch_of_questions = [data_by_levels[level][q_idx]['problem']]
batch_of_answers = [data_by_levels[level][q_idx]['answer']]
print(batch_of_questions)
print(batch_of_answers)

num_questions = 1
num_trials = 1
['There exist constants $a$, $b$, $c$, and $d$ such that\n\\[(\\sin x)^7 = a \\sin 7x + b \\sin 5x + c \\sin 3x + d \\sin x\\]for all angles $x$.  Find $d$.']
['\\frac{35}{64}']


In [ ]:
config.n = 4
config.max_depths = 7
config.num_batches = 3
config.num_phases = config.n**(config.max_depths-1)
# config.batch_budget = config.num_batches**config.max_depths
# config.batch_budget = config.n**config.max_depths -1
# config.max
print(config.num_phases)
print(config.max_depths)
print(config.batch_budget)

import logging
logging.basicConfig(format='%(message)s', level=logging.CRITICAL+1)

In [ ]:
importlib.reload(mcts_search_v51_tree)
importlib.reload(mcts_search_v21_tree)
importlib.reload(mcts_search_v71_tree)
importlib.reload(mcts_search_v81_tree)
# importlib.reload(mcts_search_v51_mini)

In [ ]:

trial_idx = 0
config_name = f"mcts--n-{config.n}--d-{config.max_depths}--normalize-{config.normalize_embeds}--level-{level}--qidx-{q_idx}"
torch.manual_seed(100000+trial_idx)
torch.cuda.manual_seed(100000+trial_idx)

for question in batch_of_questions:
    agent = mcts_search_v81_tree.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v81_tree.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)
        

In [ ]:
print(tree_dict['0.0.0.1']['text'])
print(tree_dict['0.0.1']['text'])
print(tree_dict['0.0.2']['text'])
print(tree_dict['0.0.3']['text'])

In [ ]:
stop

In [ ]:
with open(f"results/tree_{config_name}--trial-{trial_idx}.pkl", 'wb') as fout:
    pickle.dump(tree_dict, fout)

In [10]:
trial_idx = 0
config_name = f"mcts--n-{config.n}--d-{config.max_depths}--normalize-{config.normalize_embeds}--level-{level}"
with open(f"results/tree_{config_name}--trial-{trial_idx}.pkl", 'rb') as fout:
    tree_dict = pickle.load(fout)

In [17]:
config.n = 4
config.max_depths = 7
# config.num_phases = config.n**(config.max_depths-1)
config.num_batches = 5
config.step_budget = config.num_batches*config.max_depths
config.num_phases = config.step_budget

In [20]:
importlib.reload(test_mini_mcts_search_v55)

np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for q_idx, question in enumerate(batch_of_questions):
    agent = test_mini_mcts_search_v55.MCTS(config=config, question=question)
    _, _, agent_completions, c_depths, c_phases, c_step_cnts, p, ndepths_arr = \
        test_mini_mcts_search_v55.mcts_search(question, agent, config, tree_dict)
    
# for question in batch_of_questions:
#     agent = mcts_search_mini_v51.MCTS(config=config, question=question)
#     tree_dict_v51, tag_list_v51 = mcts_search_mini_v51.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)


-> p = 0
selected_node = 0

-> d = 0
0.0
   q-value = 0.9004
   u-value = 0.0000
   puct = 0.9004
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.1
   q-value = 0.9917
   u-value = 0.0000
   puct = 0.9917
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.2
   q-value = 0.9946
   u-value = 0.0000
   puct = 0.9946
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.3
   q-value = 0.9839
   u-value = 0.0000
   puct = 0.9839
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
selected_child = 0.2
selected_node = 0.2
step_cnt = 1

-> d = 1
0.2.0
   q-value = 0.8989
   u-value = 0.0000
   puct = 0.8989
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.1
   q-value = 0.8672
   u-value = 0.0000
   puct = 0.8672
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.2
   q-value = 0.8931
   u-value = 0.0000
   puct = 0.8931
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.3
   q-value 

In [23]:
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")

ncomps = 0
nvisits = [[0], [1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]
total_nphases = 4
c_nphases = [] | nan (±nan)
c_ndepths = [] | nan (±nan)


/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rco

In [ ]:
print(tree_dict['0.0']['text'])
print(tree_dict['0.1']['text'])
print(tree_dict['0.2']['text'])
print(tree_dict['0.3']['text'])

In [ ]:

for question in batch_of_questions:
    agent = mcts_search_mini_v12.MCTS(config=config, question=question)
    tree_dict_v12, tag_list_v12 = mcts_search_mini_v12.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

In [ ]:
print(tree_dict['0.0']['text'])
print(tree_dict['0.1']['text'])
print(tree_dict['0.2']['text'])
print(tree_dict['0.3']['text'])
# print(tree_dict['0.0.0.0.0.0.0.0.1'])
# print(tree_dict['0.0.0.0.0.0.0.0.2'])
print()
# print(tree_dict['0.1.3.2.1.0.1'])
# print(tree_dict['0.1.3.2.1.0.3'])

In [ ]:
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for question in batch_of_questions:
    agent = mcts_search_v51_tree.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v51_tree.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)


In [ ]:
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for question in batch_of_questions:
    agent = mcts_search_v71_tree.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v71_tree.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)

In [ ]:
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for question in batch_of_questions:
    agent = mcts_search_v81.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v81.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)

In [ ]:
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for question in batch_of_questions:
    agent = mcts_search_v21_tree.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v21_tree.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)


In [ ]:
# for idx, node in enumerate(agent_completions):
#     print(f"\n-> idx = {idx}")
#     print(node.state["text"])
# print(agent_completions)
# for (key, value) in tree_dict.items():
#     print(f"\n->")
#     print(key)
#     print(value)
print(tree_dict['0.1.3.2.1.0.0'])
print()
print(tree_dict['0.1.3.2.1.0.1'])
print(tree_dict['0.1.3.2.1.0.3'])